# 04 深層強化学習（DQN）— 状態が連続で、表に収まらないとき

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: Q表の代わりにニューラルネットを使い、状態が連続値の問題（棒立て）を解く回です。
> - **たとえ話**: 盤面が多すぎて全部を表にできないとき、「見立て」で判断するようにする。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-5（DQN）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

> ⏱ 所要時間: 説明を読む 15分 ＋ 実行 約4分

## ゴール
- 状態が**連続**（小数）で、Q表に収まらない問題を、**ニューラルネットワーク**で解く（**DQN**）。
- DQN の2つの工夫（**経験再生**・**ターゲットネットワーク**）が、**なぜ必要か**を、外して確かめる。
- DQN が**不安定**なこと、**簡単な方法（ルール・離散化）に負ける**こともあることを知る。

## 前のNotebookとのつながり
02・03 では、迷路（状態が数十個）で、**Q表**を使いました。状態が、台車の位置・速度・棒の角度のような**連続の実数**になると、状態の数は**無限**で、表は作れません。
そこで、**Q(s,a) を、状態 s を入力するニューラルネットワークで近似**します。これが **DQN（Deep Q-Network）** です。

## 題材: CartPole（棒立て）
台車の上に棒が立っています。台車を**左右に押して**、棒が倒れないようにします。
- **状態（4つの実数）**: 台車の位置・速度、棒の角度・角速度
- **行動**: 左に押す／右に押す
- **報酬**: 倒れない限り、1歩ごとに +1。つまり、**リターン ＝ 何歩もったか**。**満点は200**（200歩もつ）。
- 環境も自作です（Barto らの古典的な CartPole と同じ動きの式）。

## キーワード
- **DQN**: Q学習の表を、ニューラルネットワークに置き換えたもの。目標 y = r + γ max Q_target(s',·) に、Q(s,a) を近づける。
- **経験再生（replay）**: 過去の遷移（s,a,r,s'）を箱にためて、**ランダムに選んで**学習する。連続する遷移は似ていて偏るので、それを崩す。
- **ターゲットネットワーク**: 目標の計算には、**ゆっくり更新する別のコピー**を使う。目標が毎回動くのを防ぐ。
- **発散**: 学習が壊れて、値がどんどん大きくなること。
- **離散化**: 連続の状態を、区切って（ビンに分けて）、有限個の状態にすること。表形式のQ学習が使えるようになる。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from rl_envs.cartpole import CartPole, heuristic_policy
from dqn import train_dqn, train_discretized
import run_phase4

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]

def rollout(policy, seed=0):
    env = CartPole(); s = env.reset(seed=seed)
    ths, done, trunc = [], False, False
    while not (done or trunc):
        s, r, done, trunc = env.step(policy(s)); ths.append(s[2])
    return np.degrees(ths)

rng = np.random.default_rng(0)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for seed in range(5):
    ax[0].plot(rollout(lambda s: int(rng.integers(2)), seed), lw=1)
    ax[1].plot(rollout(heuristic_policy, seed), lw=1)
for a, t in zip(ax, ["ランダムに押す", "ルール: 倒れる向きに押す"]):
    a.axhline(12, color="r", ls=":"); a.axhline(-12, color="r", ls=":"); a.set(title=t, xlabel="歩数", ylim=(-14, 14)); a.grid(alpha=0.3)
ax[0].set_ylabel("棒の角度（度。±12度で倒れる）")
fig.tight_layout()

**何をしているか**: CartPole を、2つの方法で5回ずつ動かし、**棒の角度**の変化を描きます。赤い点線（±12度）を超えると倒れて終了です。

**出力の見方**:
- **ランダム**: 棒は、**20〜30歩ほどで倒れます**（平均25歩）。
- **ルール**（棒が倒れる向きに、台車を押す。AIなし）: **200歩、倒れずにもちます**（満点）。
- つまり、この問題は、**ルールで簡単に解ける**のです。それでも題材にするのは、**状態が連続で、ルールを知らない**エージェントが、報酬だけから学ぶ様子を調べるためです。ルールは、**満点の基準**として使います。

In [ ]:
df, curves = run_phase4.run_all(verbose=False)
df

**何をしているか**: 6つの方法を、10個のシードで比べます。数字は、学習後に**探索なし**（Qが最大の行動だけ）で動かした平均リターン（満点200）で、**最後の3回の評価の平均**です。評価は、学習とは別の乱数で行います。「最大」は、評価の曲線のうち最も良かった値（**これは、後から選んだ最高値**なので、実力ではありません）。

**出力の見方**:
- **ランダム 25.5**、**ルール 200.0**（満点）が、下限と上限の基準です。
- **離散化した表形式Q学習（2000エピソード）**: **182.1（±19.4）**。連続の状態を324通りに区切るだけで、**10シード全部が満点（200）に達し**、最後も安定しています。**簡単な方法も強い**のです。
  > ⚠ **注意（Phase6 の訂正）**: この「離散化のほうが DQN より強い」という順位は、**このシードの組での結果**です。**最終評価（未使用のシード）では順位が逆転**し（離散化 158.9・DQN 172.1）、差は**誤差の範囲**でした。正しくは「**この問題で、DQN と離散化は、同程度**」です（06 を参照）。
- **DQN（再生・ターゲットあり、300エピソード）**: **151.7（±51.6）**。**10シード全部が一度は満点に達します**（最大200）。しかし、**最後には、平均73も落ちて**います。ばらつき（±51.6）も大きく、**不安定**です。
- **再生なし**: **10.1**、**ターゲットなし**: **9.5**、**どちらもなし**: **9.5**。**どれも、学習が崩壊**します（ランダムの25より悪い）。2つの工夫は、**どちらも必須**です。
- **学習率を下げ、ターゲット更新を遅く**した DQN は、**165.7（±21.9）**。ばらつきは小さく（51.6→21.9）なり、最大から落ちる幅も小さい（73→33）ですが、満点に達したシードは**6/10に減ります**（最大195）。**安定と引き換えに、頂点が低くなる**ようです。平均151.7との差（約14）は、ばらつきと比べると、**はっきりした差とは言えません**。

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
xs = np.arange(1, 16) * 20
d = curves["DQN（再生あり・ターゲットあり）"]
for row in d:
    ax[0].plot(xs, row, color="C0", alpha=0.25, lw=1)
ax[0].plot(xs, d.mean(0), color="C0", lw=2, label="DQN（10シードの平均）")
ax[0].plot(np.arange(1, 21) * 100 * 300 / 2000, curves["離散化した表形式（2000エピソード）"].mean(0), color="C1", lw=2, label="離散化（横軸は300に縮尺）")
ax[0].axhline(200, color="k", ls=":", lw=1); ax[0].set(title="学習後の方策の成績（探索なし）", xlabel="エピソード", ylim=(0, 210)); ax[0].legend(fontsize=8)
tr = curves["DQN（再生あり・ターゲットあり）|学習中"]
sm = np.array([np.convolve(r, np.ones(10) / 10, mode="valid") for r in tr])
ax[1].plot(np.arange(sm.shape[1]) + 10, sm.mean(0), color="C0", label="学習中（探索あり）")
ax[1].plot(xs, d.mean(0), color="C0", ls="--", label="学習後の方策（探索なし）")
ax[1].set(title="DQN: 学習中と学習後", xlabel="エピソード", ylim=(0, 210)); ax[1].legend(fontsize=8)
for name, c in [("再生なし", "C3"), ("ターゲットなし", "C2"), ("再生もターゲットもなし", "C4"), ("DQN（再生あり・ターゲットあり）", "C0")]:
    ax[2].plot(xs, curves[name].mean(0), color=c, label=name)
ax[2].set(title="2つの工夫を外すと", xlabel="エピソード", ylim=(0, 210)); ax[2].legend(fontsize=8)
for a in ax: a.grid(alpha=0.3)
fig.tight_layout()

**何をしているか**: 左は DQN の**学習後の方策の成績**（薄い線は10シードそれぞれ、太い線は平均）。中は DQN の**学習中**（探索あり）と**学習後**（探索なし）。右は、**経験再生・ターゲット**を外したときです。

**出力の見方**:
- **左**: DQN は、**80エピソード前後で、平均がほぼ満点（約183）**に達しますが、そのあと**成績が大きく下がり**（160エピソードで約115）、また上がって、**上下を繰り返します**。シードによっては、**満点から30以下まで落ちる**こともあります。**一度満点でも、そのまま保てない**のです。離散化（橙）は、早くから高く、**安定して満点に近づきます**（横軸は2000エピソードを300に縮尺）。
- **中**: 学習中（実線）と学習後（破線）は、**ほぼ重なっています**。探索率が、最後は0.05と小さいためです（崖の迷路では、探索が大きく効きました）。探索率が大きいと、両者は離れます。
- **右**: **再生なし**は、10〜20付近で、**学習が進みません**。**ターゲットなし・どちらもなし**は、**ずっと10付近**（紫・緑の線は、10の位置で重なっています）で、**全く学習できません**。
- 満点後に成績が落ちる理由（推測）: 良い方策になると、**箱（経験再生）に、倒れない良い遷移ばかり**がたまり、**倒れる場面の経験が減って**、倒れる状態の価値を**忘れてしまう**ため。（「破滅的忘却」と呼ばれる現象。）

In [ ]:
s = torch.tensor([[0, 0, 0, 0]], dtype=torch.float32)
rows = []
for label, kw in [("DQN（再生あり・ターゲットあり）", dict()), ("ターゲットなし", dict(use_target=False))]:
    r = train_dqn(CartPole(seed=0), seed=0, episodes=300, eval_every=0, **kw)
    q = r["qnet"](s).detach().numpy()[0]
    rows.append({"手法": label, "直立のときのQ(左)": round(float(q[0]), 1), "直立のときのQ(右)": round(float(q[1]), 1)})
rows.append({"手法": "（参考）理論上の上限 1/(1−γ)", "直立のときのQ(左)": 100.0, "直立のときのQ(右)": 100.0})
pd.DataFrame(rows).set_index("手法")

**何をしているか**: 学習したネットワークの、**直立の状態でのQ値**を調べます。報酬は1歩ごとに+1、割引率γ=0.99 なので、**Qの理論上の上限は 1/(1−0.99) = 100** です（ずっと倒れない場合）。

**出力の見方**:
- **DQN（フル）**: Q値は **約100**（上限の近く）。**価値が、現実的な範囲**に収まっています。
- **ターゲットなし**: Q値が**数万**（84,000 など）まで**発散**しています。理論上の上限の**800倍以上**です。
- なぜか: 目標 y = r + γ max Q(s',·) を、**学習中のネットワーク自身で計算する**と、ネットワークを更新するたびに**目標も動き**ます。さらに、近い状態は似た出力になるので、**Q(s,a) を上げると Q(s',·) も上がり、目標が上がり…**と、**追いかけっこで、値が際限なく膨らみます**。ターゲットネットワークは、この**追いかけっこを止める**ために、目標をゆっくり更新します。
- **関数近似（ネットワーク）× 自分の推定を目標にする（ブートストラップ）× 古いデータで学習（オフポリシー）**の3つがそろうと、**発散しやすい**ことが知られています（「死の三つ組」）。表形式のQ学習では、起きませんでした。

## 試してみよう（演習）
1. **学習の長さを変える**: `run_phase4.EPISODES = 300` を `600` にして、`run_all` を実行してください（約7分）。見るポイントは、DQNの「最大から落ちた幅」です。分かるのは、長く学習すると安定するか、それとも上下を繰り返すかです。
2. **経験再生の箱の大きさを変える**: `train_dqn(..., buffer=10_000)` を `1_000`（小さい）と `50_000`（大きい）にしてください。分かるのは、箱が小さいと偏りが戻って不安定になり、大きいと古い経験も使えて安定するかもしれないことです（私の予備実験では、6シードで、50,000のとき最後の3回平均が約170に上がりましたが、ばらつき±49の範囲で、確かな差ではありません）。
3. **ターゲットの更新頻度を変える**: `target_every=200` を `20` と `2000` にしてください。見るポイントは、Q値の暴れ方と、学習の速さです。分かるのは、更新が遅いと安定するが、学習も遅くなることです。
4. **離散化の粗さを変える**: `src/dqn/discretized.py` の `EDGES` で、棒の角度・角速度の境目を減らして（粗く）、離散化した表形式Q学習の成績がどう変わるか見てください。分かるのは、**区切り方次第で、簡単な方法も強く（弱く）なる**ことです。

## まとめ
- **状態が連続**だと、Q表が作れない。**ニューラルネットワーク（DQN）**で、Qを近似する。
- **経験再生**と**ターゲットネットワーク**は、**どちらも必須**。外すと学習が崩壊する（ターゲットなしでは、Q値が理論上の上限100の800倍以上に発散）。
- DQN は、**満点に達しても、そのまま保てない**（不安定）。**学習中の成績**と**学習後の成績**は別。**後から選んだ最高値**は、実力ではない。
- この問題（4次元）では、**離散化した表形式Q学習（182）の方が、DQN（152）より安定して強く**見えた（**ただし、最終評価では逆転し、差は誤差の範囲**。06 を参照。**1回の実験の順位を、結論にしない**）。**簡単な方法・ルール（満点）を、必ず基準にする**。DQN の利点は、**次元が高く、区切れない問題**（画像など）で出る（未検証）。
- 強化学習は、**シードでぶれる**。10シードの平均と、ばらつきで判断する。

次の Phase5 では、実問題として、**家庭用蓄電池の充放電制御**（合成の発電量・電気料金）に、これまでの手法を当てはめます。